# Launch-Deets: retrieval, evaluation and usage

Run top to bottom with the project `.venv` kernel. Local checks are free of provider calls. `RUN_LIVE=True` enables the controlled retrieval example and one independent agent request; their token ledgers stay separate. Never interpret missing usage as zero.


## Shared setup
Run the following setup cell once before either testing or the walkthrough. Keep `RUN_LIVE=False` during dataset inspection.

In [ ]:
from pathlib import Path
import sys, json, re, time
import pandas as pd
from IPython.display import display
from html import escape

APP_ROOT = Path.cwd()
if not (APP_ROOT / "src").exists():
    APP_ROOT = APP_ROOT.parent
sys.path.insert(0, str(APP_ROOT / "src"))
from launch_deets.config import load_local_env
load_local_env(override=True)
from launch_deets.config import Settings
from launch_deets.data import load_archive, validate_archive, split_documents
from launch_deets.retrieval import HybridIndex, structured_lookup, graph_paths
from launch_deets.providers import jev_check, jev_score, pinecone_upsert_and_query, pinecone_ingest, neo4j_paths
from launch_deets.agent import ask, SYSTEM_PROMPT, _row_evidence, prepare_runtime
from launch_deets.evaluate import evaluate_local
from launch_deets.trace import RunTrace, ranking_rows, ranking_metrics

RUN_LIVE = False
import os
if not RUN_LIVE:
    os.environ["LANGSMITH_TRACING"] = "false"
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
RUN_INGEST = False
RUN_EXTRA_EVALUATION = False
settings = Settings.from_env()
if RUN_LIVE:
    print(f"Runtime initialization: {prepare_runtime(settings):.2f}s (no model calls)")
archive = load_archive(APP_ROOT)
assert not validate_archive(archive)
index = HybridIndex(archive)
question = "Which component was tied to the Charlie yaw incident, and what fixed it?"
filters = {"launch_id": "L7-CHARLIE"}
demo_trace = RunTrace("controlled demo")
agent_trace = RunTrace("agent request")
ingest_trace = RunTrace("optional ingestion")
evaluation_traces = []
display(pd.DataFrame([{"python": sys.executable, "records": len(archive.records), "documents": len(archive.documents),
                       "chunks": len(index.chunks), "chat_model": settings.nebius_model,
                       "embedding_model": settings.nebius_embedding_model, "dimensions": settings.nebius_embedding_dimensions}]))
local_report = None
agent_result = None
evaluation_rows = []


## Dataset testing â€” local only
These optional checks are for validating the dataset, not for running the application. They make no provider calls and do not upload data. Run shared setup, then work through AÃ¢â‚¬â€œD.

After editing a CSV on disk, rerun shared setup to reload it; no separate reload cell is needed. Saved outputs reflect their last execution and may be older than the files. Use the heading collapse control to hide this section when following the walkthrough.

### A. Golden questions
Inspect expected facts, sources, tools and development/holdout labels. These are test expectations, not agent answers.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(archive.golden)

### B. Source metadata
Inspect records, document metadata and graph evidence links. Missing optional fields are not automatically errors.

In [ ]:
print("Structured records")
display(archive.records.head(5))

print("Document metadata â€” all documents")
document_metadata = pd.DataFrame(
    [doc["metadata"] for doc in archive.documents]
)
display(document_metadata)

print("Graph relationships")
display(archive.relationships.head(5))

### C. Validation and chunk metadata
Expected: no validation errors; 17 documents produce 32 chunks. Offsets identify the exact source substring; multiple chunks from a document have distinct offsets.


In [ ]:
archive = load_archive(APP_ROOT)
index = HybridIndex(archive)
chunks = index.chunks

print("Validation issues:", validate_archive(archive))
print("Total chunks:", len(chunks))

for chunk in chunks:
    if chunk["metadata"]["document_id"] == "DOC-L7-005":
        print("\nStart index:", chunk["metadata"]["start_index"])
        print(chunk["text"])

### D. Regression check: GQ-003 major-incident launch count
Expected: 3 distinct launchesâ€”Alpha, Charlie and Echo. This check exposed the AI-generated golden answer of 2. The CSV has been corrected; this cell calculates the count independently.

In [ ]:
# Check GQ-003: count distinct launches with major incidents.
# This exposed the incorrect golden answer of 2; the correct count is 3.
major_incidents = archive.records[
    (archive.records["event_type"] == "incident")
    & (archive.records["severity"] == "major")
]

display(major_incidents[
    ["record_id", "launch_id", "severity", "summary"]
])

print("Launches with major incidents:",
      major_incidents["launch_id"].nunique())

---
# Application flow and evaluation walkthrough
The dataset-testing section ends above. The sections below explain retrieval, the agent and evaluation. Hosted calls depend on the setup flags and can consume credits. The application entry point is `ask()`; it does not require running the optional dataset-testing cells.

## 1. The complete request flow
`ask()` â†’ NOUL â†’ LLM tool selection â†’ selected tools â†’ evidence-only answer â†’ citation-ID check. Empty/error tool results may trigger one `retry_node()` call, with duplicate calls suppressed and four total tool calls maximum. No cross-question conversation history is used. LangSmith records the complete request.


## 2. Prompts and history: what must survive
Routing receives system instructions, the question and tool schemas. Answer generation receives only answer instructions, selected evidence and the question. Optional retry planning receives the question plus concise failure feedback. IDs, dates, revisions and uncertainty survive; old routing messages do not enter the answer prompt.


In [ ]:
print(SYSTEM_PROMPT)

## 3. NOUL: safety decision before retrieval
`providers.jev_check()` â†’ `TypeSafeClient.system_one()` with a NOUL question asking whether the request bypasses protections, obtains secrets/private information, or misuses the assistant.
If `probability >= threshold`, stop. A low probability allows retrieval; it does not certify the future answer's accuracy.
The local teaching examples below inspect only the synthetic corpus; hosted retrieval/reranking is gated by this result.

In [ ]:
defense = jev_check(settings, question, trace=demo_trace) if RUN_LIVE else None
allowed = defense is not None and not defense["blocked"]
display(pd.DataFrame([defense]) if defense else "Live safety check skipped; local-only mode.")
if defense and defense["blocked"]:
    raise RuntimeError("NOUL blocked this question. Stop here; do not run retrieval cells for it.")

## 4. Structured evidence: rows, dates and provenance
Production `lookup_records()` calls `structured_query()` for explicit filters and lookup/count/max operations. The cell below is the simpler lexical `structured_lookup()` baseline for comparison; it is not production routing.


In [ ]:
rows = structured_lookup(archive, "Charlie IM-03 C8")
structured_evidence = [_row_evidence(row) for row in rows]
display(pd.DataFrame(structured_evidence))

## 5. Graph evidence: links are not an answer by themselves
`trace_relationships()` â†’ `neo4j_paths()`; missing configuration or no graph matches can use `graph_paths()` over local CSV.
The entity filter is `LAUNCH-L7-CHARLIE`. Cypher follows three typed relationships, not arbitrary model-generated Cypher.
The agent attaches the supporting CSV records and their timestamps to the returned path. It must not infer a pre-launch fix from a `verified` status.

In [ ]:
paths = neo4j_paths(settings, "LAUNCH-L7-CHARLIE") if allowed else graph_paths(archive, "LAUNCH-L7-CHARLIE")
print("Backend:", "Neo4j" if allowed else "local CSV (teaching example)")
if not paths:
    paths = graph_paths(archive, "LAUNCH-L7-CHARLIE")
    print("No hosted paths; showing local CSV fallback.")
display(pd.DataFrame(paths))
evidence_ids = {rid for path in paths for rid in path["evidence_ids"]}
display(archive.records[archive.records.record_id.isin(evidence_ids)][
    ["record_id", "occurred_at", "event_type", "summary", "source_id"]].sort_values("occurred_at"))

## 6. Document search: the detailed branch
This controlled example sets a Charlie filter explicitly. In the agent run, the model chooses the query and `launch_id`; an empty launch argument means no launch filter.
The Pinecone namespace separates this corpus from other uploads. The launch filter narrows results within that namespace.

### 6a. Metadata filtering and BM25 keyword ranking
Compare eligible chunks before and after `launch_id=L7-CHARLIE`. BM25 applies the filter before scoring; case-insensitive equality is used locally.
Pinecone receives an exact metadata filter `{"launch_id": {"$eq": "L7-CHARLIE"}}`; use the canonical ID.
The actual hybrid function requests up to 16 BM25 chunks when its output `k=8`. Scores express keyword relevance, not probabilities.

In [ ]:
eligibility = pd.DataFrame([{"launch_id": c["metadata"].get("launch_id"), "document_id": c["metadata"].get("document_id"),
                             "kept": c["metadata"].get("launch_id") == filters["launch_id"]} for c in index.chunks])
display(eligibility.groupby(["launch_id", "kept"]).size().rename("chunks").reset_index())
sparse_unfiltered = index.bm25(question, k=16)
sparse = index.bm25(question, k=16, filters=filters)
print("Unfiltered top candidates")
display(pd.DataFrame(ranking_rows(sparse_unfiltered)))
print("Filtered BM25 candidates")
display(pd.DataFrame(ranking_rows(sparse)))

### 6b. Query embedding and Pinecone dense search
`pinecone_upsert_and_query()` is a legacy name: it **queries only**. `make_nebius_embeddings()` â†’ `embed_query()` â†’ Nebius text embedding API â†’ `index.query(top_k=8)`.
It sends raw text, not token IDs; the vector must match the index dimension. The result is chunk text, metadata, IDs, and similarity scores.
The trace captures embedding token usage before LangChain discards the usage metadata. Vector arrays are not stored in the trace.

In [ ]:
dense = pinecone_upsert_and_query(settings, archive, question, filters=filters, k=8, trace=demo_trace) if allowed else []
display(pd.DataFrame(ranking_rows(dense)))
display(pd.DataFrame([{k: e.get(k) for k in ("namespace", "filter", "k", "read_units", "seconds")}
                      for e in demo_trace.events if e["stage"] == "dense"]))
if allowed and not dense:
    print("No dense matches: inspect the index name, namespace, filter and ingestion before assessing hybrid search.")

### 6c. RRF fusion: combine rank positions
`HybridIndex.hybrid()` deduplicates by `(document_id, start_index)` and adds `1/(60+rank)` from each list.
It does not average BM25 and cosine scores, which have different scales. A chunk appearing in both lists gets two contributions.
The same document may have multiple chunks. Only the top eight fused chunks reach the reranker.

In [ ]:
fused = index.hybrid(question, dense=dense, k=8, filters=filters)
def chunk_key(c):
    return c["metadata"].get("document_id"), c["metadata"].get("start_index", 0)
bm25_ranks = {chunk_key(c): i for i, c in enumerate(sparse, 1)}
dense_ranks = {chunk_key(c): i for i, c in enumerate(dense, 1)}
fusion_table = pd.DataFrame(ranking_rows(fused))
fusion_table["bm25_rank"] = [bm25_ranks.get(chunk_key(c)) for c in fused]
fusion_table["dense_rank"] = [dense_ranks.get(chunk_key(c)) for c in fused]
fusion_table["recomputed_rrf"] = [sum(1/(60+r) for r in (bm25_ranks.get(chunk_key(c)), dense_ranks.get(chunk_key(c))) if r) for c in fused]
display(fusion_table)
assert all(abs(c["rrf_score"] - score) < 1e-10 for c, score in zip(fused, fusion_table.recomputed_rrf))

### 6d. TypeSafe reranking
The first three RRF candidates are scored with the current relevance rubric; the best two are kept. Scores represent relevance, not truth or answer confidence.


In [ ]:
reranked = jev_score(settings, question, fused, top_k=2, trace=demo_trace) if allowed else []
rerank_events = [e for e in demo_trace.events if e["stage"] == "rerank"]
if rerank_events:
    event = rerank_events[-1]
    before_ranks = {(r["document_id"], r["start_index"]): r["rank"] for r in event["before"]}
    comparison = pd.DataFrame(event["after"])
    comparison["rrf_rank_before"] = [before_ranks[(r["document_id"], r["start_index"])] for r in event["after"]]
    comparison["kept_for_answer"] = comparison["rank"] <= event["top_k"]
    display(comparison)
    print("Prompt:", event["instructions"])
else:
    print("Reranking skipped; local RRF results are not TypeSafe scores.")

### 6e. Evidence passed to the answer model
Production `_document_context()` merges duplicate paragraphs within the same source/revision and preserves facts, dates and source IDs. Ranking scores and offsets stay in traces; they are not answer confidence. The preview below is an inspection view.


In [ ]:
evidence_preview = [{"source_id": c["metadata"].get("document_id"), "record_ids": c["metadata"].get("related_ids"),
                     "occurred_at": c["metadata"].get("occurred_at"), "text": c["text"], "relevance_score": c.get("jev_score")}
                    for c in reranked]
display(pd.DataFrame(evidence_preview))

## 7. Does reranking help on this example?
Use the labelled Charlie development question, GQ-007. Compare source hit, labelled-document recall and reciprocal rank.
These metrics use the expected **document IDs** in the golden file; labels may not list every relevant document. They do not prove the answer is correct.
Comparing RRF top four with reranked top four keeps the output size equal. Reranking may improve, preserve, or reduce these measures; report what actually happens.

In [ ]:
gold = archive.golden.set_index("question_id").loc["GQ-007"]
expected_docs = {sid for sid in gold.expected_source_ids.split(";") if sid.startswith("DOC-")}
rankings = {"BM25 top 3": sparse[:3], "RRF top 3": fused[:3]}
if allowed:
    rankings["Dense top 3"] = dense[:3]
if reranked:
    rankings["TypeSafe top 3"] = reranked
display(pd.DataFrame([{"stage": name, **ranking_metrics(candidates, expected_docs)} for name, candidates in rankings.items()]))
print("Expected facts:", gold.expected_answer_facts)

## 8. Autonomous agent: observe its actual decisions
Now run `ask(question, trace=...)` independently. This repeats NOUL and any providers the model selects; its tokens are separate from the controlled demo.
The agent may use structured, graph or document retrieval; it is not required to call all three. `answered` means an answer was produced, not that it passed factual evaluation.
Local fallbacks are recorded when optional dense retrieval or reranking is unconfigured. Other provider errors still surface for diagnosis.

In [ ]:
agent_result = ask(question, archive=archive, settings=settings, trace=agent_trace) if RUN_LIVE else None
display(agent_result or "Live agent skipped.")

### Actual calls, arguments and preserved history
Each `tool_calls` event shows the model-selected tool and arguments. `retrieval` events show the actual filter, sparse/dense candidates and fusion for that call.
Expand a model turn to inspect exactly the public messages sent to the model. Dates and citations in a result remain evidence, not instructions.

In [ ]:
timeline = [{"step": e["step"], "stage": e["stage"], "elapsed_seconds": e["elapsed_seconds"],
             "details": json.dumps({k: e[k] for k in ("turn", "mode", "calls", "filters", "backend", "reason", "blocked") if k in e})}
            for e in agent_trace.events]
display(pd.DataFrame(timeline))
for event in agent_trace.events:
    if event["stage"] == "model":
        public = {k: event[k] for k in ("turn", "messages", "tool_calls", "response")}
        display(HTML(f"<details><summary>Model turn {event['turn']}: messages and tool decisions</summary><pre>{escape(json.dumps(public, indent=2))}</pre></details>"))
    elif event["stage"] == "retrieval":
        print("Agent document query:", event["query"], "Filter:", event["filters"])
        display(pd.DataFrame(ranking_rows(event["fused"])))
    elif event["stage"] == "rerank":
        display(pd.DataFrame(event["after"]))

## 9. Evaluation: local baseline and answer review
`evaluate_local()` is a legacy category-directed source-hit diagnostic, not agent accuracy. The full LangSmith experiments use labelled golden answers, source/citation checks, model-judged factual support/completeness and human review. Explicit abstentions are evaluated against answerability labels rather than scored as unsupported claims. Judges can be wrong; evaluation tokens are separate from request tokens. The final section loads the current comparison report.


In [ ]:
local_report = evaluate_local(archive)
display({k: v for k, v in local_report.items() if k != "cases"})
display(pd.DataFrame(local_report["cases"]))
if agent_result:
    display(agent_result.get("grounding", {}))
print("Review the answer against its cited passages; valid source IDs alone do not prove semantic correctness.")


### Optional: evaluate more live development questions
Off by default. Set `RUN_EXTRA_EVALUATION=True` in setup to run GQ-004 and GQ-012 as additional agent requests.
Each row shows expected facts alongside the generated answer, selected tools, latency, citations and reported tokens. Review factual correctness manually.
Holdout questions are not used to tune this example. This is not a model-judge evaluation.

In [ ]:
evaluation_rows = []
if RUN_LIVE and RUN_EXTRA_EVALUATION:
    for qid in ["GQ-004", "GQ-012"]:
        case = archive.golden.set_index("question_id").loc[qid]
        if case["split"] != "dev":
            continue
        case_trace = RunTrace(qid)
        evaluation_traces.append(case_trace)
        start = time.perf_counter()
        result = ask(case.question, archive=archive, settings=settings, trace=case_trace)
        evaluation_rows.append({"question_id": qid, "question": case.question, "expected_facts": case.expected_answer_facts,
                                "status": result["status"], "answer": result.get("answer", ""),
                                "tools": result.get("tools_used", []), "seconds": round(time.perf_counter()-start, 2),
                                "reported_tokens": sum(r["total_tokens"] for r in case_trace.usage_rows if r["total_tokens"] is not None),
                                "calls_missing_total": sum(r["total_tokens"] is None for r in case_trace.usage_rows),
                                "review": "pending human review"})
display(pd.DataFrame(evaluation_rows) if evaluation_rows else "Extra live evaluation is disabled.")

## 10. Optional ingestion token measurement
Past uploads cannot be measured retroactively from this notebook. Leave `RUN_INGEST=False` for normal question answering.
If enabled, `pinecone_ingest(..., trace=...)` re-embeds and upserts all chunks into your configured index; stable IDs overwrite corresponding records, but embeddings still consume tokens.
This does not create or delete an index. The usage appears under `embedding.ingest`, separately from query embeddings.

In [ ]:
if RUN_LIVE and RUN_INGEST:
    result = pinecone_ingest(settings, archive, trace=ingest_trace)
    display(result)
    display(pd.DataFrame(ingest_trace.usage_rows))
else:
    print("Upload disabled.")


## 11. Token ledger: actual reported usage
Use the per-provider ledger for complete request token counts, including NOUL, embeddings and JEV Score. LangSmith automatic totals currently cover chat calls; custom provider usage is also saved in root `provider_usage`. Evaluation judges consume separate tokens. Cost estimates use dated list prices, exclude infrastructure/subscriptions/credits, and are not bills.


In [ ]:
traces = [demo_trace, agent_trace, ingest_trace, *evaluation_traces]
usage = pd.DataFrame([row for trace in traces for row in trace.usage_rows])
if not usage.empty:
    display(usage.drop(columns=["raw_usage"]))
    totals = usage.groupby(["run", "provider", "model", "stage"], dropna=False)[["input_tokens", "output_tokens", "total_tokens"]].sum(min_count=1)
    display(totals)
    print("Calls without a reported total:", int(usage.total_tokens.isna().sum()))
else:
    print("No live usage captured. Local-only mode makes no provider calls.")

## 12. Save the inspection results
Save only the synthetic question/evidence, public messages, scores, usage and evaluation diagnostics. No settings object, API key or full vector array is exported.
Inspect `raw_usage` in the JSON if you want provider-specific details such as cached/reasoning token counters where reported. Those details are not added again to totals.

In [ ]:
evaluation_rows = globals().get("evaluation_rows", [])
report_path = APP_ROOT / "reports/trace_and_evaluation.json"
report_path.parent.mkdir(exist_ok=True)
report = {"question": question, "live_enabled": RUN_LIVE, "agent_result": agent_result,
          "traces": [{"label": t.label, "events": t.events, "usage": t.usage_rows} for t in traces],
          "local_evaluation": local_report, "extra_live_evaluation": evaluation_rows}
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", report_path)